# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': '92f838cddfc00ca9563f6a9a9e198d7c6fc5c6514382d719e4c719101c3b0289',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_olmgswZ1yWfvloCpH@g;7#l`O}0IzH{#S+>)eUgbJ6B~c}|C{n{qmE{_(HGgnk=5x;1%nzJjIQs&C00^qej(d78?e5qV'
            '32fXpHufEdhlj7?x{Ri2EPnW5y@@YG5ohzb7;KVFoF-W;mT|g?i&B)!sEFrcx)sOwCtPTv3JsIZHk*px)iRka50VVul_JVS{LgiA'
            '8KrSnh2n9ZM$<t86*pKrtHdndY(>5h)iNnXnqMTdK2(~E%eY7u+ew<wqI9^~9xz;ytT%a4m0WzC&ui%PJSpN?m2Sl%FW^HgsFECn'
            'DA)Na7FAqU{o%pEiyUfZ7cinC7WF30qj@Qy+NPeSNx7sRZ>h*oJgY>W0oWqOLJV|~q*P`uu8IVJ9+<CdYA_P;CWYxM>&+%jV7~IY'
            '+SHZUMAdS5aCmrluqg7im`oORRTuGOB1Z*av%HF`B+tr&1Nm+lmCH1ls-J%;^GrP#v3e?(b(N(0S3TVn`7AC=^`=_l7+_G`@;s{I'
            'Dp|*>UG+N@`1glAi@6+j7}{3Feuke12e02edh^Tc$)6rQ|K;gxF&5Vc0{$Gv?*T)gc2-5{VJHsg@nxLm8+<X@q|q#1<|)h$ek$V2'
            'B)$Ut0mJ}wIt&>?w1Hk<;%{-7WM#crB(nsoL-!(lxrktv=`$3Z*E5``2@Y%vAa>BRU%r3y?9KD1lV5*)_WI|iuO>geeDd@;^)aZD'
            'Y#W3kDA%wy_^^)WNxjCWG%7CQ!R4I+EXFhs0MO${KfihQ^2OxMk57MkI{Eq4%O9RSH^v&w<3&`bRWOn-=q2<Cv0ws-TSCX+<4M?j'
            'UF0y3ELz9#K6n!UCHhk>UPoCOxMj-Z1D5&b)c6p=P%q=jEKl=d5|t&w_{SfQ;m;57#{@BrN>8~TesIgJqxX}myqIH=yUzO=Y;t^m'
            '=h*sCTqN1#65gMlnjZki<6;KQ!j~VSw2X~!X<Su6A<9iO1KfiGvf_^a2`*qhCi!9l=m|Ji-ZY=5UGw=<RLtTutxNIxap!~{|IH`#'
            '^ufnY=$oS6Kc7PrWK{q{&5C408w2KVy+Iu9=@L}xNffX1OuT{iqhcFpp?I~7;%r+)=DR;7Wn64a^VLt=GAfb?c8m@1$Gk3Mv*52l'
            'YvUp>L-C}DE`XR-u)3l6BlP)gP>Sb3J>jth=UIW>B2<9(qjU?6VNYV%pj$uyUApfsG2%%J7hXReyoxVss64EyMH?-a(doVWSmXH4'
            '@!h-8^!~lm(+AVzd-v|neiz-Foy7O=%onG>o8OJ^J&3+Jd2sKW`1IuT!SP8HExx(`V7{229N(YL7soBob&(?6WmRp;(a}*6T@5di'
            'YFSTVo{AZi0HQXW<?Ex1JikceBdk;&t)nD6$`>g-$PtvYqg#w7O=fWhNNP^P%O9Q(tM`?!*k|mVr<Exjfk2~yDyI=`F+4)BJ=i=k'
            'z){d7kiqSSsGzDbd<1m6kCO0wDgGF5;J9V><UJhHvjjPj^#%|e4%nk2Pt$2MTZx?Z(oj6At9%V@&cq*Hz7Yi+=*UE5`IUqRlUEY!'
            'vCe_7fCIQnsx%(p;Y56uNuh+KY$BS%aX0jm!U=5f4{`d*C(L86-a!lRKlt0z!f!q|Ej$^X?xPDlBZEI734;~6fhH{O&L7;F-?=+K'
            'S=^s}bNA%rcX#5u(QI*d3PfS?yL)#|;(O8kZ>IO7i0H=r_&EAzKK<r2x*H!a?%e&{G=b6KhzIcx_rAZnSXTdV@5$em7RXM0G8wQ2'
            ')wwbsxB0z&+7WyBheyE7z5D^Vx>qk>Jbf*>yFWht^RF*oJwfiS*K9?WNSG^ugK9}f9K8c^gAO%(U8yq<e_Y8^PqrB(c$_f|{5)@t'
            'Li$OkFg_LX^d%Sx%z3ehkwrb|9~>M2rwMF+nWwlzBR<%n59_#x+%vW~=iehyRt1b9h=5W7C|bSu)dc_2--5mvd<){ztQs-PjglV$'
            '2ccyiL(}UdOG+ejvN27oNFLCnlA1)ZNg)XeH3(7n3YvpP-Ckl9?qPojpOa0mKTPwhxajq%da^Jee4$1&(#T+a2U$rZnea2~Ba62v'
            'B4B5br>NtnMUfXaON4(Ghu<e4v94oSO+9!yv)8)KFN)nmap<YMjxT2@;n8e?><5)f5Sik6&)>cN&W6(0zvODqLlisVEP{#D%>#z0'
            'CE)pFo+t*cyBTkm7a-#Rp`TAA63DeMQGymCuqPk?z!^7N(&}YU%ATg)D&8VlCox++!vfr}uVBrL!tLG}+#LRSf$ZNwdtLyH?3}Z-'
            '?fGiS6`@!JvJazc31!#tllJ^|0aZdDEJBv^^iL$v3Be5H4gfJSGmmX&4*<z(7St@oO&Wz{ADl9^5_?Nx0r(+oaS8AVH+l#Dohj(&'
            'a{70Y*vK=GPLRo>PSYNeS)v<&Lp)3K<N|sRb0T#UL#`vy=n75{;G5!%Y9fJ~0^I>#YZ)OSZzdYiXtu1g72-t##HW`=>*+ii$&y3l'
            'o_hE1-Me$YABriUtG!6vjbXh(^+k^$_01)a-<R=we(EWL3dablT`YP!!Q-RHk1#i|&^&)YNh3g)$vi3A#Pg?Ug_Kj%NB|+C%P2{S'
            '=OfKfx;3;}Fv?PFk}QKA_WH+11DLeo0Rj7-4>`eS8O$N@oX;{*0&4?oBta+D8LGTuI)&o*z_`JYgi4Yq6@Q=;nqd8a2O(*sL{f@*'
            'ewCq?3)N8B3|?hbP5_=1aEK0&%Y{!;S4Lw6ZkH+FT+SwaLgfo8!Rk3`#9(D()U`#}M^x>>%Mw<8M#?p40nU?Tv4{&)@!`A-)tu44'
            'YR1OGobfY_yXRC&aWPO#ei&ZFRnOt=^vQwefqFYK=qc|<t|fT}A~MXGmAJIct1_>FQ%AO<+`<lB6JDTB2w`bdfF7YTtk>X%{e8Gz'
            '!QZd}aQx+%SXWpvI9ez9ihgp1IiO{dk;V@|HsFu+K$yV4-^4Q{AiU9<`8c!q)^qnPHNX#(lBTT3+ST?pH_GZ`L5#<4+c!)so--yS'
            'XOig{(sO1_H|>xFj|Kd$W*o-(7ifM`<awngeZsw-^oNp&0TaP9E&rjKifzP>@q{&ij+*)NbE@ZR22Ch8PN7-&c5}#j+ac<}Pp#^q'
            'xpvJV=<vdw<Klg?pK}h80WgnOr+715o9ue~EbD|84qKgRi6&tqn)I7~FhbyYt3YeSAlm5Tc%*t7q!~Pn<l%atKf^AD@4%<KTPzq}'
            'tKRH}icU6cK~-$+k<-a$w=#sEeAZbT5(TSs)~4rmHNJa~4nRQH4Tt~^uVHm@NLGc(bX&z`PnG9`(3m8-U~LT4a2oXcVh{S+?D4BQ'
            '!wxfp93&ac63z}EYx1~)<K&{K;hc`jL`)-82$so*!0uWznj$9Ef6t4|-Z{SYJexACG1aHXH%v8gvY&{hw_lz3{R}lsPw5||?$9R%'
            '8@{!>wJG3W?k$2}{`~0Gvk}jEnv_viiWTfMoNl!uXt@CS!T`5(7NFTgia(er^w!(SV1GS*ZS@h*tyxw&$x4?n*h^od0dUppAKZXw'
            '9%1TDypCuFYTyJvBt;s*QbhAa^sen~Mk{o_>%(l|EDSxO+2IdSu}TvH7l|@10bRvKJOj>X8*P#l;SL>HObNTQgM-eU<~=U=?KRKP'
            '7lAQSlrq#NTfv(GIDNE`VrSKo8;z<Z|GcL#KkOXE!(xF(;mH)h6-lhuowNtodMSEM5s<YOuu9VyfAojuFA(6iVj2^xr5UyyMk)UN'
            'U;mA#dL3QFNiJJt>ub48Hl(PPFR2^bkZUvW5IQapaiIm|Kw}~y2P_Q>X5>`$Br7Xm%dLI^GR*SznwrDT4aH-!;Fe-tgP>EFv0_G`'
            '0<J{_i!BY(3n*C%QWgW-3k(SI?BbnlZIHa{Tih}nV)Ysgv4rY&AJ_)3bt&>#v*+YMR#elCvOOQj)E!`s=@Up?AYIv2k;199jNiv{'
            'Ynpv%!=i6Cs&y>2lC4S*tC+(8*15nPgOXiah=fr-rLwSUWIg9qB6bZW!+1T7=V&Lw6-z+qX)u6cUKG)0iSX2H_JD>DweGzf{~lp|'
            '3n=(F&w%Pxyh>#nk(nPB3!1r#in&m)kgMbVmjo~pzXHw8uRw?*S&3E70yW^7=sh0ccpkrfJL>BtGqf$t1gZltc>5NQlA}gif%BzD'
            '>JRa)IH9KT*V)NAGK|5<P<lzA^!fN)d@TUjD1d+I&FSdeTpfHTmttBcDbf2X0YP+P%vuknNTLHF?38)69HCekie;Q!EGzkAn%2dn'
            'OfJ?Dy?`Fz+3GBzA^`rySrpL)Isk|sY{Qg25RNLKgek7;v_RPtmW4HK1Tu&2;_jR+BcL9*Wzfw5;H+VN&W=Zy=PbZRLXAuO`u<1d'
            'U;zShj8gO&RhXZ%(8+@h;Iu@8!K}zP0ja-mfCW%$#EOdo=}=%xsi<;7aT^!Jtlg4<>_cpENF4=4O5^c0&F^k>&2zgO0n*LlNnRuu'
            'K*v&GYB$8bBcUEA*$mY=ad_we#AaKd)F=-fZJF6^&lqpn`%RhvH+43>n3Tor{A|JhK6HDg3jEba8(!d8#)OdL>jpV@&;w7m7QSu`'
            'PWN_b9&?AFvUi|33+ITTX|x^pejQx(k4{Iu9|ueLw`(z4n&-vgv{5um-3nvqHU9HOu@k*%yaMtNXCq;t9q&kcG5d#oYqLl79Lc@0'
            ')0a_cj21tS#5B)S5HRc63bx?{_*^)3=JK_dUv5-xP=_ZASX5vqkxZab@Di96)RfD2a6nU`%>MAH)MK5zM|uHx8J&xs8jYBx$wttS'
            'xYyEGi<C|?paQTD+N1BYba8e<uyjZJOjgb!pd{x*g8(3x;h95s4*`B4L>}(=+MR$M1J+R8xxdAhY=9dxj>r#B9&Fv;8+hZdCO<@K'
            '`)A0{%;R%>fMxJ*3#_0E;$K9((V7knWI0CPvl5-Z#>d0sdn4Xz*9|7^Mp0D(MvvheP;VIA_|Eb185V&3)NW9IP(6y^FCZmA{>$;{'
            'XkPP3$8bcW&UZA2odZ)pA>WvUKAi$|V>yhy>Q4SgP5dWJx;1??@4t00+Gbd1lGQU<b9v%N#YH(n;j%h|-yl1G6(VbUL91n(Ula$D'
            'S13Zt?o7&fCU>;>7iQ+i^bhZE?R+?V3@89RQ#}8fUo`u^hD|SbJZ;CpRt%PaIY8R73<%;NMQ!$FoQfEUCh$7g>7jUq)U>pusK(}i'
            'N^R8X*XW{JRzO$_(W_u4W)bwEk8DPkXM+!Mk)tlQlB)M#mq3WWfBEJ|@lMUcJJAD@luygJ0BX<Bhch6!70CI-tG#Pa$vfyHGcgyy'
            'EE01}oH+0|SDAP>Xn@14$I%)&epDSW+X?$+9?u5=C(q{Ppv(u02pITMBIbsSi6Xj)P=2AIt}=uxIUm)Lp?HHV6M&9DaLj6yh1RgA'
            'U`~*fhEgV~$ngaP#5GLKNG$~jMitENGQR@32SghQJ0S{4{;m~{i+EM&08If?UdevN*#*oLG|qjS#tZa@;dMz<HQz9sPyvn`RCD4-'
            'P0^*f1E>W%%d$`e<AyZ^;pqYmav<peVia37k{FJy6~Tw<RSb<TqszEt`v_WJO4*MVZ>JUnTOlk4M8m0RFD-uYy#4)H95<wUwnnq*'
            'nJFFx1M)dFQJa1%?8YC2;PCJjV)*$%qYuhwC`mNC2@tslZG;NPCp_!YxPW}WVOVH}I2$rqL;OMo#qsDSjm#oYlD5+CfoKK$MO-`J'
            'Nby4Nc(<JmfN&p=&P{ccq^Vd++E9MeG?{%5^p?T`=!QNzp?X=4m|WVEO6vzL6W!{ySChNEK_#O-??02?k!Bu-i8@B}<FlWihLYH{'
            '-tXt7^ojuwRWHypYkbm|?qrgBq4^FLC?ne)84qz=u`~BbQcj0bCbaq4%ARf|xOly(w%AOC!nPE@0G|MrfDEjmPMqcS#gc^L1a=bc'
            'N}N#I>vL!=%~8K0g|si_L=gD``UpZ-el?U9Fs8o~=#6p4;EJCgNR!t>OlVZ5{;;J7HUSNz4Mq#}XmKR2Nv*5&W9Zanj<B~`{?8!8'
            'e*gUG@$re($R;XF*(;jnf((+<Retz}o_gry6IWyIGqBC%xMSMzxGMB3Dk`Z|v4(@%42q&iGNFe6(`OHO6~hWmLUD<r0`!AIDwL1R'
            '20+t!-wVb;%MJill`~8)ApQXM@}#D1p?|K+z@x84AB-fc<C~T8>jZ?5Qg0yYrdQ|P$U5E4HE+IDhYzA*DS{VAj{@`HSR@6UcO+a{'
            '(7qJ!jG?_lq~{*;0g7`W0ng)dg9igbLxu4}I(ZymFoh-HTv2SPT!~iaI!h|4S&Xsg((Mu+@rf+gL)#&NS)nzcHqn<*P1=z-nE++P'
            '|Axo*q$v_6*$a~DdULWxpY%h)za=z$ft?x$WfttEE^B}oUBu%>AOm`?5#(riyx1LG6Eu1jWhPenJOzB0Shc`&I5n2vMA+dhh57Bd'
            '#GOu-u`GENC6!S^ZH~X>I4RObb5BizBy(eHx1){=klK7j_CDC>jgMHziS|*anRAf&&oJ-$80+zo3)EhiW;OP&j66v9u-AM>HKRfU'
            'uR5EQT%Z)T%z~CZLxC%$1Fflb+yq%tNBuUOM(<=jfxOz4f+l}0@bIbLiU0Nb<qINy9QTBfJp4PZr=tg7L}|GdGM0-tJ9a+#_kaC2'
            '8Cq46W{_F`2%T=Udp4@$l`@_eet=@zTx;f`i*-6jw$j$;Gj0KMjVyV-poYko1q01ggqf=uFsE-H5-LqP9KmVL#;jq3U>-(K13NRC'
            'K;Guk<QW5N`kNQL0`xmy;6E5OMz|pL(&r6w?&l`M152;CpP%g5eRAY#`?);hskPrEnq!BD6r9&ng`tBAi+M&alSr8})T+0IT|YX1'
            '53<i7UM$eQLhCPNC$}C8j$lK=JGqrY!f9z}V)=~J)F^hSGlxwP&_Txp(<dGbZBj9`1Jahz(?eD*DI$5=CT0&RN>M9HucB>9Vr<z+'
            'wYWhlDgP-$BmtyoN95&D)Oq<vAi|J5S<=>m^&g6ts6NwDBRtoc4qqT=tzMS1XY&)uHje`4YY;GwiN#tO_lbQ>`-~H%9GBMj(Q05c'
            '=>|HHCgep2tj-$NLeB`158IA^9Vk>EiL>hfQE)b?wi|XJ0_h=VC+ay_L6fZd7wOf-I%MPBE)>JzP;G-iZWut#K*<DbC*{IBD)t5`'
            '0S2j}%(yAgh8(j|5Bj7HigN9#6Gz;xqMC>gMSKyz=c5*l%*%QmWlfj}8-x%?pq(ZHQU|tz96q3N9^n(IB99KV!zy`(9;pF{*)pE3'
            'qyP<w4IEp7o{}<d7r07SAZCiDH_#0PjSp2#&mK0Um*-&~v4uTaC)>QZJ-IXGLRTV8elro%EQdjZScs5<fD#GGrI(zqhh-cUvt_Sn'
            '5dUrYRqyTm>;9<uyZ7Db?GS%`*EddG1W60#h4-3w%$HDY1pl4=@3-0cSKs{!%a8bn1B&(gw+Hozl+`UzjT+h-tkQ#!!>Z9~^S#*v'
            '`Tln7mx~>FxsA4o$XE52@W<f5PVT7_K`t3{7bHW<o*UgWmFJ~*ZhGJS{)@NsKCII9X?Snff3Kf<-{A+C83W!BO)Hz;vEk7}n;C$j'
            'i;iwYmOJV9nI%<i6R(Q%qN__P;IDsrbA34hN({b{@X@~})srSPeT*B6IJ#{sR43T=oo&LACAzyUq-x*;-&K?-5@mH2!-JC4#!d~M'
            'm(mm>?8cA);7k(0zW7Sqxqo~tzNXog(DChke8<A7Y^pCiuYG2k(?mXnT}Of0h273#LzsVTF&=w3>#7F6{fxcKQ9Gj9lAp0gDc)hp'
            'cR;w24^hX;6{@MVb9pFo^eMT*{bqTfFp+J0p$zrm3v;)~C)_%De`oVPLN8+VQ>mS`{|6CBht&$a+ITj?0Tz%#T`7#z7@`XO!zy&0'
            'F$dGvDrHS1kSif}CIW;hOUGeGa!bHrAXy?GoeY)2k^tdYLezxO7vL){F5<$1Z}Rb89=@dl(atZO@{?TPu$i`QtyD`3MT(G!G;f8{'
            '#8HP$5k}7rDclZh_3DibvDrU+B_%q$48ko@I&6I2OkWcH(sU~E>nvc?dW}mUTnz64#faW_e8!DGJoZSdH8GiGi~$m&j6fvcDt1Vj'
            'P$Fr3$9YBfPnS6*hCAYYfGZTw18F#qM8L2DE(uMbYbKE48m-d!340_917-*FxIzJz5d!w#EUn42i9N`ae8hY2Dq@yxbi?-ROP-pz'
            'QpSEAiNW?F<RE9rVlnMtIR`htn2`vK>>i%PGVV``3b^Ue89YZnEVXad_H>SN+NFT0WX57`9g?QTKEvDOsFORDf(k{nLBA<~1W%co'
            'awtG=ZDy6{pCfcIEyqr(mT8)s*MY=Wo?02?v6Cc{PIL+=pnY**06q)I(UaX=1JqvI{{QWwAcB#S-BXyxL$A;((Zj$;io+J)isJ*5'
            '_4c-H*q7+D3U|7;b0?sS0`o9D<vJM$f8qn*`D8;dV#{WEf$9zehOzy!GMT2a@r~zLi7;*GyM8-lM6PZFL~f{&9A((-nUPvn84+Sp'
            'X2fQFiZN5~Igu^OCCQ*Ot!M5qa2-*0XqXW0#hHUsSWZ44+{&4HPkS^Ot%9wk-CxvN>h7YR<?b%rUjBAjXMR{kbuQU8n1RoW1&vIA'
            'D8Rn^jT-Fi+?2cw`Jp69nydrrEo{D$>l^$oMeRt&`rMGbJ{c5q&*uj>E-bo*THWpwow#w8v7Xh(zC~XwkPR|kg#xLd_4VxdoCm?;'
            'L_h5KETxNn8v`2LqOL81>#N=S>!lHKdjl5suB@mj7Qc{;F0J?6gT5ys=vG*62{NV-vshOaVrg~ibt6JS8UCn+gniabqZO7&pn9{d'
            'g7cOEv-bvYhdAocFsSuAq3XWtMKC!g8nk+GaP!XAfXS3xHPXM!si6ymE)35ut0sf^XrBYz2!rH+E$UtNY4=>>?vdvj85BzuN7JSe'
            '{BUa`aqy)We7-*b?X~nPM+IptN8GzbK0L)cts_hiaqy+~SqVUL`sKys)zcSGp1u+r*wr!cbf34|Z1W-Xos<QeOMwenZ<I*v^~`J}'
            'b6C$FaM)o>jy44PBF2Oub26R-7rf5rs$576Iu|y@{<0~lD~BMjtCDG&luBDd=CvMG8qgbj?4x{ta(sMjV+#<PMlUcrKArDI*QQdy'
            'ctud?cX-VWj*jnOP5{%Jb_NJJK{YL)GI^qwJ@p`c8<5dFX9<N8+bB=@yQegxO&KkkGyu*}{}2^GR{~@t!T|M^fmBS3-h%LD03t%U'
            'LEgUFf}lJUuhB76h9_iOA;Agq5R@S%x;O08x~|hI*`RV679U+^P&NSUMOg&~s6}D?(dgLdaHDV;p}XuxB*2F=?)ACyR4|>7$d?M!'
            'j~wjvIIv<NBn{!owCiDCwQh=MZp2;SALnQoBQBp_5_5(*TCjg3>d3CYzbBNlNJd%=$ph9H5CEMR1Q&Nk(+=RK-4hU8O%}7xRw4g0'
            'Ny)$H49nOBCRABXtAuo5yI}INa_f51(C&O-463Zx7)C9N9$S7WDh&q>)tO-IgqmUe-fw@V-ha*91{D!sixYQJ&Gz?}MJvCC`fcaR'
            'v#}&Xvw;Ks$efn8vkAdw!N8nT=!dnKUlH_nbuYphOQrEmH`Jh8AOiMz;ItGD=C#*cgYcKzhdgQ)Z?j!|z4z@FfA=MsMbMH?ay%@Y'
            'EIoW_6Gg4=Y-|K<8nkFkGJhW;L`2FsgS9OvnMX}W1@?1%VsJeKKn;uZ!1MQpp5FSrIK1|rd;nXcLVpLx?YCl2L7JHU?y#c~D#IkO'
            'qlEHKYcvfqiWqD4g14ERX{rQ!nLL4$>|z|$)nf1<P&RLs+;JLHT%8VcYhoCrP{2AlrK52YKpAJ%WP^bvb=;g;H#jT`c@|ADP9-HB'
            'h6b$Cs-{tPQAZc@gXJB56Z9lYDvVU-yp=}IDt&go(Nj<g->6w<=Nz0wdj@({`Q%KKiws@D;drE@UPle`z7a1oI$fo0T=b+4TBx+e'
            'vn`JXg^C(82~re_h^yJK??i;sIYQY2ttFweSy@E&i99gWW3cf`5uyKBDSg-Eh=Y>#9P@_t23C*Up&KPQY#-?(P$<2>+*ER;6OQ#Y'
            'Vizr+MKn(HOit8H%6QaY`R7lF1Wl3!!U<R}^J}k-FqqfkGMUBWV793#P1$-=<7NPHXB@zsJP52Kml%%;CI<?|gmjl<Zfe-<2Rm2>'
            'se%tVU~5sTCXERyEz4yzml0EAJH?<p&gSHJIiZZI(D%G9X#vMlX`wKFtR6x!1v!~UJHBT_vr&B9vwx$Sm%=D~>95pWAoVoZpP@s`'
            'ht?@n4vFnCG!^EIp1aIio6+mE{@Z(OnwLzUR+xMi{U8A|=XKi+{6@aoL!ZcEVYN4){fp&;+v(ig<O+80m&u37&1fNdlu@An(B7U}'
            '^S!E5Snni_X3=VBjI%@ONCxSSVES&YL4(Gu=bU_`qSLz#I$CIHDD1}DiV#=(f!KRuxwr1OdU=~h04;XBXA@4DxlZ!u`1362aoS_C'
            'j;kyY+kDkGSZB%fpvrwL?pxp|lFY)UqSnrmQ3C8?Q{$YO6x=%n2B7pn7FG-`p4Lstz7e~Dh)aph#=c=5Y4+Xdg2Tm2S<ITTX4@yg'
            'v+73l=I>$GPx{Wc4vx-_s>3mcO)|EI5i@(JtDL*rnopL%yHx@||1#fZ(K?x_By45xHq6Tn9E3_?_{j{;ZP}b#f%y%r&}|taHd(Sz'
            '*PLr&gL#JG9UI9BmvDvUl<Vx4tgpgI2W3f&85DByobmt}5X=}gp!dt<Vws}Km2O9PX*Cc5o5a{4?TAs?$|<U;C)pzjq^9H)>Xm7i'
            '0NdR6--tE&Z^@9Hw`jKUN@|jLksTJq$t)&6wdGc&ex&cr^Yk^{vp=Fc%MJTWvq!RuM>k~ZN;GI593D95-qc4r_XifF()zG+v|}DE'
            'nUF2==**ZzdK*jUUCZC@#z-E<I^qzSZ{WFV+UbYpE1WxDAF=N5#|+8ey412z*S{8Uia%&)^}^ZHOY4QA*PGS8vgdc2rRMj<<KF35'
            'WmDlNt<w=^T&T9XzA<9sGfp+~FQK^W0ln-02Xf2u`2Qqs+3luB(<e`DR3pig+u^SzeB%svsuIwy+9YuQMiw29mURMrcAVlpHTzlg'
            'dHq4We)UG&$el|GSY=1+RH<0A)ybH6X5Yq}_8Bww96JX-rokZI4vn^d`Y3a2d3nGp!EaQSx3mLE<ti*<;+TX&!#HD`hS)IEztKe0'
            'hRxQZmc7VR*D@NZ3TV*@h11z~g%ef2u^CY(ccW|Bq_h$tJbD~nn?UJRV9q6HjOuAt)t*BYwgsmGDeK*gQ2+1${-3~jd5!OS=9?Y9'
            '>6>rfQrdW-HG||YtFsx?)@1plZKRT~B|qGKOtuVKcaHb@;hmb-(71o5MI5ar7tzLb?Yn=kF_JYMZA@i<z;P*Eo#ZUYqE#;Qf?Q|W'
            '6LzzNL$QgWQ0(n2!t+M4knB{9JEEYR#H>X(6ww`)5!9}86*oQ^_%QO8l;^${u+D(7Su3S5i&mJ}3@zOZ2n`S<ZmL4({r%1YT$VM_'
            'Vjy2iU}wkYjW*i?12gVGP{mp>1}EnVz7w1(`Ahg#qKiO<nRU=yVOEqDz(NHpV`0?9_Ikk(G4%`l>pT2wgnvB@oLre4kvAbc(YG*I'
            'L=offxy8P;6ZKA*1Zkx1w;2}<yT+Q<N}*c})_>5{Cq(zx*GuuOG=<+&{F1vF{Pl4SzEx)Q6IIvR5&nw)neVtVK;o%6k8LiEH$Y>z'
            ';OX>Oy_)@T;Jj=b2YtI^PeYOzatX_XBrl;Mfe47IkACq9jJaykXp0%cC+`(?IyruDypQMCHMY9O?@rOHK_wB@!nOZqw)P)=n~nnb'
            '%_$N{zQ#u9%RAlMD-S{T65c%^+`D`9IJ_75cxbbAqmNa)b*)ePkG`FUbB8I%s(n@LTYJpYz~jdKE24&Y^!hb=6r-rgQ3@1dK%v2m'
            'TbLtsldqIY0W-VCDP^P}Ku;qKszCtg7ad2Ga4_aW#5yd<RfqG;u*HS%u%vvCbs_SF<YYOo48kB0f<3UK%RHHLwL`Ue<(UO!ciZSr'
            ';`yP77mIkt;mi4=(bTM>BBHx0qYF+(dBG|e@i^K%uB!nBO0M%uG$!%Na}O+C32!c#(oVL(U<Jnd$w0kvLljN-WCrX8dl{MSf+vz`'
            '#}!LpeHHiq0XX+e%%k-MGVt4cr6RkQoIk`=bOvXyS(W=NTjb~FJwW^({o+RA^6v2X_-o_qBKR5h_^Y`5r;#8BpN2^EcfN9ecFS$8'
            'a?o0o#`EuCYU1KiHh;a0=K0kKFw45J@z)~SyhhKQkvKKqo6l!|y8E@DLZvmHA4r)#5-%dWcoQ#e$n+EIqkb+DASv6&gnyxUO&qB4'
            ';$>YSf~ud5loossGKzZreV&52{0!LbWEP5FkPXIK01wXVRSZDj5l8z^^4~whzq~&~@dy(#*`R4kXgs<HP=V$L$}6!i#4FE$8a`JK'
            'uhhezLh)4ExUGee^Z1Am`&Vcsyw>!27t&&Bs@sPbHtm8JE-~DNfdQT=BK`^g@yBl-$j8A+cufBTxq$w=<|EpsQ~lA!4$Fl1;J;G~'
            'EF3)We|9sI<R3bB(uoXDiIRB{;d!)03FS3wJ)WYHLrycJ$i4|Bf)a{1vA*+)Pj5*$Etd->3@8u15BZ0!^XHA;w-eaN;>J1cspQ@i'
            'Ae~!yN{y-=9BZ=RX3&fweoxGs4=h`{-&YeF+{A8$Dx7IBUCc2MriD2XZj<Flw^^P!;Lf{Q^Y(>99q_F%P(_Q~Vuq7$8|atf1vv!('
            'LW}Aux1}RE+9=LssC45>3Ig1}grnR84O4h?3e=4#=|D@j4~?r|{(>QQM*be`En}ev7|7-Mr95mG5i&gL=p1ljf!VBH*D15$Weg`g'
            '-uH#~;0zD^8c58&<3TqmJF#XmY)|0;Kb(A`?$fJz^)IohgzfI2qpag7wTVZyZpyA|L*H&|Y6GaFwbsr|Lt^fF6q~I1JdRCnm2In>'
            'qlKgqV@pS(TlCI!GYa(brw9Lp+f$+>jZUs}5nNxteLHKP5C8b~?et`~yVl*^1vdh5P8|utU#~q%KUQ>)N%zd5AB4KD;=u;tsnI=E'
            'i^C^Ll;#&`EQQm|)e(}VVApz&=7L@5yAIDd#O3)KUN@cg-hFqIf1KnMzm`MkQyz5GgtZ;!{MWEaei+j@1tXKvqjiD$W`2{cblgQs'
            'K@9o@LTFt_>g6NsyVi_LB|Hi8mmht>+Tu2Mt1WK2R&CjRquSEEOzmuRa{t`9DME%ou)?{w%&W;dXKNqc;jV5}E8~;{c=)4!O|@i5'
            '6kQ;I2ThU8W1sE&=LDm9<sA%|o(2_zalB#`p-Py7+m#YUkF2YH72r_E4uLpxWi%b~%P5P|ZJCr41Yv&QtA5G8K}i`8w7M+5;>)6%'
            'c>CEhFXK$sHX@SNl%os#tUPk*S!tC*bwilC(YvoB?tg&szR0T|Q2(xzXk}zBT*#zZQs-6jtz$xl{mhecB?bddYPII08sy7a8b<{)'
            'cR6LG=46tgbSR8uWU-lCi@C{YS|pTR*Kj}3Dp$Fl>$r*#l^tY|F?)RT2)XgdC|-LZ@N8<}a3#Bh-0<zC@>Ok@YO;*L5>QOj*hNsY'
            's}o&LwhI`4i(0aj>MQHCCpWg)D(C>Xp>d~}%JGMaQM@TZC7T2i(Fow4fricG0L5sHDO7V{P)lFe!}hodoZHLL9iD`HK5}iFNlCZ4'
            'WI$_ZBq0XZS5nQ|@856-?BWn4>Lg82)e?!|n&xB|0Qa&@$|!}ccafwcaZRAVD0V{=hliM<49<ZM^-3T^3UG?yp`|eCsnz+`n7=*x'
            'lf|Juv1s7rr(IvP7Nr7ixOc68?=WrD7CJ+{c6;LD%eS7T*xNG~K=0al1MRn0ocq{%-|nmGZ1)jB`Y_@Vy36!^SJv&Lz72)&K*NJI'
            'x>EPeVTvA`FzF82??0sz?%Crs>>+Ks@%Fb*>45K?vuUN#9WSXd&%U5<td?E)%r1ETNlUgLBsqQ13*TLVjN7dP)Mkl;7j1qZ^K%0k'
            'CeT3MBF~Z!csH9^0mGKiNn5FWk3b#f$xzr*La+zfabuLLI7_kw`VZ%Ew2eUI9tMWFze$GbOwgF|$H?r|#bPLz9o&?fieoIBdTGRV'
            'Yc_-;y7MCW3#DvdMcXV|afrBZw*nAPV=Z^$OuAkeWNV;lTzPKpHP4`Ie7xQ8TC}HixWI5*g*t}q-pYX;N<PR1@A<IfLxd#oy2XZ2'
            '5#vzbl>uJ$wn_@Tk}fG>=A3&k0%s<bS=>Wi2QAvHrnZr`>`k$61}Xga)8}jvwD(5O+^}krfOo_V*31DQ<cc}^KfnB=i%Nl&$?H=&'
            'Y_qS%#ToWpFWN6iM|Ss&oMwIrhu=+@Qc)Umes7ki-WpD-b`3ok5~$`$p`Z2LUBZ9AQv;JTmm67`l&MsGF;X)ro9KaM`-e5+=A?a('
            '9aQy%ETpNTxge~muM4?W(cc}1=Eqw&YwOY<;hC-L*lyUAilfQ{J9Q){b|dqHDrDHB*_!C7LU&g?aP>#Ww%p<WZIy1O@`loWv+oU+'
            'e74>jsyG&EyR=oxuvy;-(?o}k!naWcIG5J=a#oYu80B-9EQ9j><K4!Z6wILe^3}6HJbUrzIXn9i_oBT4%{*I4sl;m_X?Xb@`jrh7'
            'U1%;Z)F$1zRI8B=q8y56tOlRwF)QsUyERbA=g)sa5%|rU*ElL1cQGV~e7EOx=@s5rSy!AJKa?4W<K}`09z;?a4vtvA334=I1=w$&'
            '?}e60=MGGLBsIug(bu#ve3|Z-ee0wSzxCO+O><r{4quX2uh`eH=wu_LWSP9E)AVQgxhFXua{nJHkAKDxb@WO}Uei$0Vw5_;jVd<O'
            'neJSw+v}vqVT;xyLH_4DDdLGFmA-`8ZeR8aSwAsW!eN@M6T0`c;C$IGl${cpG%E}Xy`uO7yVp_&VKdBBSn2}QfTmjo;#zUlRO)=B'
            '_6jAv&XpIxI~iwz@g_Lm+2dCol$TUEjqaHfhv6c5J*H5s)%e<iy=x2S=z;*O<2GNp7X!mo$CtQ+ux1(+IK_0qQqSpHf9Kv2=FFv@'
            'b6w@k>el&zv76|^X_<4bdDC}F4A(38yT@1AtC$ed;!X0EaqkFkCUoP!h)MpUjN%C;dJq1P00l~c`pNscGt9xP^go{%qxi{OmZau3'
            'W9U~IcXC!Ot?@p6vAGH9ke;B8&h^tYGvAs@+CJx)Zui>IZ&3lC9IQz0rV7;MG`FaQ*G{S+N?k)5)+}xVT~+F|GUm_v+_Qd#_TX)B'
            '@+NJ0`G(s7rE$aWflj%+hYM31`}Hs1zj^lN`P0d-KR$c?^V3(8pI$zB`uz2+H+X=TR@2>=d_AT<u#|b4nxs_G-JSWO*gbSa?M2Ys'
            'sHt{(M_>E9pmi%C-9V4Bj>}U@!)g0HE+g@SI`l`~5#C7cd!Q<Aa%ARKW2|wS_G{G}B-G_KZh1STQ^;R-471(7j1B8%xF<T!!cneh'
            'LD|<$${^ipjPHL@YmIX=;X)L8Y^n{0cso1gk?s}2Sv2@?etn8A^n2#JKf2u<>;Z9`SqPN%saq}8wJ{Fc-C+a~m+>|d$tn@Re=pKR'
            'OyPu4n*|O4yhElIIZOzAbI-tDUS#oVgt1k4WK?iIJVd!Zi?Z=`t3}vW?pfHZpCl+4GdPyl%Ue{&Ma!mq3oj3>b<k3jL8qE`XymjT'
            'It%pI;M@aBMuDTdR!j8i28UmjpM!>H3Sma?T-nSE)h*=qwUN~f$hH9Ig=-gap=xo?nStiL52pUwnt>+KSQ{R{?}A5PJ@8mGY=M)n'
            '{m}Sh3s~c{GG=qV7bP2`e#p+teFuzfz}L<@USN(7KuBKA_NhzggqaL+qr@5B%W0qBRHElQl1*ihI1+@JWzgt>Erm^!gn=&VK@kPo'
            '4AswBvCz+SMK;nWXrqWQmJfQ4@jl1Hdmc*l5dce4n~0MK@Sn`<AcvqXRKUQ9Q;x(il~v*^3szqo9f?zTd1~>>`dWoH6#1ze%t7P*'
            'ZkUVh3D%#7>&@N3<he10V=qwBh?E3YahaqBo9(@VH#A+H2Xu2KGlhzI?wi+~*i30KsLbMuzczhT2J2{pwETE<IXb2rQ}l~ybo`y^'
            'bvl@h=q{-=nZ7^0f0qjLqT(ZF@Rq0mB@4Pl>XadFlJ}Fvn!ar7%e$w?8wYHJpQ30cJ8RFf&D+v9o!q31KQ_aO1DpM3TDNtRfot27'
            '8i{=gdO28KUeB8~-SZ?E?%(=`a8E?ALLF&6S-}V@)b*4iS|#iw|4=9!%Yfde)i(N&AO~Ed)+i~Xb@XAV_P?DXWh`zjoH)m>DN<?;'
            '7_R3_W$-DH1<G`QuI!h@c`Gj1$u!jtxNYSCIk4kW8as*;ciXK#ZZOdke7i?zXzJ+*+-_blJm|=u;A3zcg%BzIFjwAQ{BUPmtv)8o'
            'gCnC#(3;ZRA*WZ0GT3>ES)<N`B23Ke<b9zBiiH(?0V2oZQKmZMkuN<|xZnu@YVJL}*41`;SS1j$B0+Zs8)bHVh2KDrDpa^mE{bTg'
            ')UomUHRsEqk3GXq5?TjNXrCPJ+@W*h{<%{aJ4EJ+6}dJFD`zM3J{-%yz_g}qTp*RR6MEVK+aEA@&`PxsdWL4NOcNtBHcww8!frdZ'
            '4%G0V6$6Ky_PEWihfRdsxXppva1;wSZZU#pq*9xUb)l&6r8a+Rmvou$v6rQ{>)4F-*u$vX<uzY9ZZo7##J|Nfjyo7nZw8in%Bw2f'
            'tMSz2Cw_}0miI-kjR{7|vYeC$N#TlE#HOmmU$RW#sW-{H4CEs!14YKcbw|%Ml>RO_q{QmFJLaXYt&uaCP}E>?ct%$Q&(Cpng6r1Q'
            '?1F2zzq=9MKZtXVqV(BiURURb?It+q85#K0rZ+byZ9pB|XI>iNnAksMv1~f5!AMHtPIrKjW3O??UAHW>yU+?I8#fW2p%q0oHo~M_'
            '9hwpOHNK;&jZZ@vg=Dm<_V4Z^7f+Aezl4BJ>;s2AkR|BHF1W?=>2q)I@->%Q@M=tEbj$90gz!*lj_uPZbcS@UcrxM!`AS0n!NMgd'
            '6162`NG6O~U*NzN!IK0p`d&rzBKnK?S|6TOyh2-10*nHA3>xAYou70Jv9%0240IGHAO6Pc6)7K%on8;|uimXin52Wthfnqd;t;GC'
            '9<ha2u_d?MT5NG-Qzg034J2Al?SHHg1D(k)7in}sjPTbe7Rkiz1i7`$O2jb?)V!vl%&21K4H(<9oGIU?=od5FN(Wzaag>V|E!=?z'
            'Yoviw8iO7utBUX>OUPNZ0FlFE7n033b<uF@+zED5qRX(2^?P|R!vV&*T4H=M^yC~FbKZ5+MG55jP@LeM;wG!z8qArR6q+=?<^q1u'
            'tY-NYT~7W1*tTI^&Cn|`XUfs*1c$dd)qFctAXP8YAUqFP!{4eRFS9t8M;oRWhM)Tzm`K~uxkc~;xtc4W??sMRVj~Txl84e0?=sf('
            'A<4=J1LtNj3JsV+DPFOj!gQy;skcqbsA49y^E_HcAentgBZXUhH7$y(vR}24t|?FFb#P$#F4`!&23%aktZ|T@8|oEmw?8C>jjhoJ'
            '9VIY_kjjXMYDgpAr{z(_HRv*t4m&xos8k53(xEx&vnBa`81ppIH<av|LKXMJqB^d+-R(0{XQT3gndhA1c05BIVlJ7I!}3s+%va~e'
            'S;GVPJP1iAW0M{&aT1amIn<>8Z<A-O-*0EsF>%ds+wTD2IPx0-%@LfvXjXB$x3zYMOdnUTd+ky5;o#M|SM4vD`R%|BNVKQGzMPdy'
            'gZzdov`8>9xa=9xrj9<Dz5YFFS?|$wnXl!nIsJ3Q>d!fjj+i;QBSsc}h3<DnqDHQzFE`W}KoWqRnAZkrf|H0z1`v~rd|d;9c<3ir'
            'ab^>&k0n}GZ!CS9OoFN?m^>EO(HuH~9yq9GF}PS`?6Cbnro%hxY4_WyI)fyG^ABgvy{OOe0Oh-9d+uu|17YBG50!Pj-&_D*AE1%#'
            'pA&lh(;ZI_B6POzhqs#HH-YW6UK$q!8q=moUVGvi8VKJ!l+o!$i^?JryYKvcpuK$2?tyaiQ_~Q1_zAn2g91Y#)f^L^nKLaoc;l7v'
            '(<y1HYcSWPslD-*)K@|8(dPsvpVoBL2i4-?nxbB)d$g{I7`M^M+HkaKT^r=72c5?GTfkwRi$d+oPl6&t8AqKy09_DlCGw<ZZh~x<'
            'K83Qovn~B1-tl7o?kKD5ip~7_oq(EIrT2=F>4c}}u?w7>-CILrDx=SrJz%Yp`yorCEozU;#Dc5M%5IS^fQ`4o$m6uIuad<7o0Ow='
            'B}w)7sp(_{*50NSrgp(mQMnyJTHXsIaSIh{;P+!ND|cyAgO;a)S*t@K9JJgK+)5tZaNs%}*ma~@+F!*?5O9;$S66D~4Y*MOtauso'
            '=pl_qap_@1_z(cDqBgHuwLe>G6DVV=QM_eQu`)7^|K5|dIT2fjX`5PliR7$~t+WhmZ?enq9R)@qc{zjAU{RS&lSj$wR8X4oRt2p>'
            'H@za8*T&7yn#*>X6TWT=mxL6*e0-Xd!6lq(ex)T|d+Gh)j>oVV(H=t<Mwy?`>&~wd?HYs6I%RaleC>9mLxe-VuH%yfp;O{Q1nBzK'
            'Vi1){xiN~597D4gsadtv#%LCijJyP+z3M@4J(&^Y<UUq5vV*jNkg3FddkfSvG=m<Eg`dqc6p=cIdizr-PW%2DTh{AHsb74oy`h*<'
            'H$kgclj&Ao+it#$D)rI^gnl`tr#v}8<H1K$+Nx_Y|G+_1s)^ev2<@aUs55tNg1tEr3gchv^<|#SVsvVkU%C@(wgL-X*zQ{m+G7{?'
            'BqsW}2*mEl!|ryvLlXCeBF^jC3QosKOleEz_Sf#loa2-Dj!WPnF<P%lNqXm=d54nw9re@QEnvr(?ld(X><^vX8#3A5(yd}CZx>P5'
            '-N+50c0a<R4bu1$bBf?ad1*X=eIW4#*o3OsicOBa(KjYj)xEbSA0JP)K-}tSOdj9sNHS)4*#H*<4&*(M@0q+1*NXJh*?Ek>xlmo+'
            '9GA_(%@Ji-7q)(SVBJlnF7_*8vQU;7A*YOg;%hOb1V0U=uf;lfkIE$uz~h`d7|zDl61u}0Hiyz8r(11b;rKZw<spq;UE5~|<V}=L'
            'oI{+Du;KV=HDsV7=lf&L6nWro>S>ykOYB#+ttd-h&s1JfOp&}TurGl;lEhlKFyfp+i#j76lX(2-#fz735HA4JF((`<0>=$0$1Foh'
            '1-p6`^WtQ=*4hX_@0z^Q#3PvfxLm?a=zv2uU|>pJzQ3@g%O-F=bYGMDYQ9fMPpdF2SXcrTqSJU50lNjgnC9~>`s;}+3=)9$5)R4S'
            'g771L22{wYA}q8qm?b4nD>?OWv;YLWqWp=MNko<3r8!y!G!m<k$_GGTz?lV1W3T@%lu4`PczESs|M+MCN0Ul2jIVeeIJFi0q(c;a'
            'S875WbkTv%KYH;*M1)tktiCBQE_bRo(zQ`91W;DP4xw&QQ{xVWJ|&uW?2e`04=_WSpm+;85{A*Iz|yix(p1ElK!=h<F<#Hmk6<op'
            'TyLH|JzY&_-iagQeM@uq4k?=4@D%qJTSg&hIo==&s@vCj_Ta@VqD+!Hjq}wqq70g__0S|Sr|YoHnAI=E%Z<D)JD-w}RdC#}G+gzM'
            'q-R`z2y6vk?OxJ~pn-;V8>MoND+x}ujNQ_iqNnjP!l&XP2N=T;^EBp5?=b^3?!gsa1KvdFeyPYS%s8gT#F3F1iDm`+hPw1Bj^>nP'
            '8ZozA=2z4qQErk9DB<w4h2{R2k8or^3DnwaPIUsCJcIc%vTm9ot&Hi8>F^8FbZ8{3@&j(}Dt4g~c~?~RdT*P%K=f3ToKmTO*l+b#'
            'Z`se?;RbL=pF3LiciTRH&C7kqonvVV1(u(!7Yy21;Txf-_kv7cMxw5)K^T%;rZ6E%0UK|gms`4`h?4%Sqd7(@`r5!`VyzHvX_%2W'
            '$@y2@E^DRD^BH*B$&w6jb|VVBoz#h|w=NuOg{k@RRz_aC{T))16p($`9e!>^o^#OoC&ina!}zUX=iV_oG3Z8nf2~mT;ctjVH@YW7'
            '(&_%9by9My<RXbNOMC@~PJxE`G37&UrYHgyY9BUH{eI}p4aVP0gLGtGZ#VVdYbTA-jX$38GMbhu%X4#?&9skqyXX-d7zn1v$xaLl'
            '!m9<{?dQvO*h<=neKzK*ug`)O`2FeqJa`n$qg6sFA1D!_c~=5pL*hXtU<>MPU9M|5??x(JgVBYCwEWi#*#CPh+utab?T<|$l9(kc'
            'z`VbQvi)ZcWxGWb+sy%Ne2B1m5cS>a7R1sDAhQ^drZ6)fpF7O#QHH6A*64ksJajnT$-u^2Mwd0$Ir><z&^z#bp=Ey$wfYR9R=+g{'
            'q9gv{uNeFA8{!^pjQOj@J1|n)tHtlkrE1fXmi9a}wASGs9eG<T-v4b86QMbV|KRY6ugN*_Hw95t$DcoVLVfazsOBN{Iif7iB7XCq'
            'G|a-vpY*p0x2TT)9%k`BILty${Shz6pt)#ThhHQde<6C@__#tX=0l*y;@OMWk6wtEFCP8*nRxQ_56_-IdG^Y>QH|6_!0V+T7Jooj'
            '=Z(m-XbV__X?rl}^0NnO+%TTMV%UajmL~f_>!Ooea-YapGR5hs%U}BKD@s^d{);kx*ftMjl_T-|(VM5QwDn-9Js-3O0n#Nvw^9Pb'
            'EM!wMB%(cNTGA~hky6N$CO#Kwo?SGS6U@#TrwhDNmP|F2Ml?+&zzWR}(wK%ODayEs7m!eTPH8c5EFO-vqbF@05+YkX@U9otoLZN@'
            'Hi9VQ2is{{CXgFLY~+F%{u}$1<6UtzjWdD|RF~%ZW@w026R8%NXdfS2y>|;tGb|u>AUURq?dqjeD>YcoJ@(<hH@x6KXRMK_Uf<IP'
            '<02v_Fv%pw?srLpv6vf<(GaAEL>z&S+hC%Uc??^O74S*g#A?`To|}TBZs<(U=M;tc%A<9P+XW+9TK$#Lcuf4Cy?Ffe`O}}CzJ4aY'
            'fBMJg&;I=Km3tXLj{{+9Is)dvNnrZJJS!p|ZnP~pxf^U}kd8yqX<;S>A)p~%NYeMkU!&ugix#%()i(`aV=DKdc!{0Kvl#~FfRssR'
            ')E218DlhAl)z>ZHPO|pj5MJcz*xxE3$LC(9JVia9I;-1$SS|$n>&M3Q*qMC)M$tXZ<hQMf!Klm5_+KwNs5{8#a|QN%zF?k@3*d3I'
            '1n3v(=neMzrW=X_sJch1;P`SnDh8<QzWOLP_TS>V-tDd4<)MBXfAjx7PQ|~$PxzyplK)3K70*)iC9<4uwBO~6e2{OnUy}BT%A_9>'
            'UoU`*C~u($G|g-DdZSDFv>ljbITTMqv5L2$)**v{g}$C-ofX63P}g7$HP>KGQx(sp!X*@6DK&L)3s7W$Vo0(q;kZ-!{{x#$%uN'
        ),
    },
    'video_analysis': {
        'sha256': 'eacc5e1ad15c78328ebcffddf527e948bbfe3ee494aea31c5fc1bc2ad1d164b3',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;vYm?iymf&~&3S6aXn~Eh_?ZlZoa+EEP?Mz&g*eO4fz1!QRP$JY~M3EYT?6%y~RrAaC%hvv{Jr4i`KvLbgle?)T-Aw`q2L}h|'
            'g@eQ4aQIHvDqdw$T&IaFCUG8TN0ln^SZs@GvM=^BuS6m%xvtV87g=$Yu7%phrA)-?IGCL;spHagTtYYQHBAr4d?n&M5wqFySkzU$'
            'lFPWRihYc*mit&$vgBsd!Qf3%$yHJ8#Oz$`q&z5*R!S5%x!BiPl}^?GDNW)^ij~aQ+kIT_()>z%d@Z7Nu|LEm{-xx#EaPnO>o!#f'
            'S&Ds;$V{m8vs7cD)-soRGDQgw5Vzuh$i$iW@yDN_MN!I9L5Bo}InvM3c(AG~QEjD&*HsNI<Ls&^(`vg%6dP=mm`uDe04|JCCuuP`'
            '$apt6q-#W($3=Kqu?BEFJ(XS|77o=B8JyFmD4|J=(6jVfimfcAP{m+XR9lg2swn}WGJ+~U0iHs|dnwj1eUaus=NM+X5&JmJEBFt='
            'CiNjp*FewlAk7nbknq2}%8rwyOg9y_SQm#>Dp;BiKusiN4SbR#g$492;slnb6o-11rD`h`OzT)}q2n_oOOa<sQA%tS>xmMl3^pl1'
            'K?>Xe4WSpmJCyP|l{aFYr3d6boJc9Fy3B#m(tNU)`=UGwSpQh4s;prpN;w_8fJx(mMV1aD0lW!a)uG5$ikuDfqp@&GK;9*jS51Fc'
            'z|d78l2q-cgW+&E*p$UyESH-afiIUL-5-jw0%QR?@vIsQ^s`c$-&DOil*L*qWu8>&UNWphTy3*-WdQvQfAS0DV{B&Ln7_x=`E!w5'
            '@Y`4+2=gZ*ZIQ_9{oyELCGx{y@czT|4<Fwze|i4n$5-#gLfj4n{4<oFU>~G7<2W0R#V~>O%ZdX&SRS%?Ew@FM$P!=i5-dyk?{y0M'
            'dpKqY@d4&{jem<F@KU|mr0W#B<6_{$O$<v)uc2X56XP#&>5l;74lw)p^UrVJeR%b9`Tp(4cQ0Noe|i1#)!XH(|MB6~oA<BZzHzXf'
            '?hjuhpy|H2=AS$Mc_UYQ`X#<j`Nh}Q41JKu4X$^QUCX5^>T)evCPnTa%Oov90=Q3baTdejNKC$?m6<aNC=&@7@nD)CVDnQ4aRYlE'
            '#F$_LAv?taxjDzJ2E2qj19-m9Vap(>=kVwgZZg;ozx~!4`?uc!Vy?u^R_4G(2WTmisdx*+mN%*5@pE}2RYl<PBA>|pp*m7ySbx|V'
            'j98k7@L{$99(Jvg8Au|8`@x4+;%0^hD5}amhy+!EWRLcm3lS$Xw-HOMwccwWh+(^eAi-7)%WPDf76?Sxak#}<BTNlh>1VoWF9H)D'
            '|FPs+(27K@rX$-F0Ku-bJ*J>~5Z`FjKqJ&*q~!PT2K3AKMV%+F%Cab<&G6siV;=7`1y^-e!&KuWor_xo^KR%3XNOWC45AVISfXMP'
            '-!8<gYtUcftd`xQ6{=?#rlcK;`AAa%*0hr@@O(6INYXS%qQDQACJ&^?E{j!B);?Vtz=$$|v`Se4Y8Q4C;scbs^b4@3ixco0A?4B*'
            '+cisQ-yqx3E0Q{&rde?#p(jGTSEzIV@z?D;F{3t@F7cWPH8g9s+d`1+`MpSUSOa_%-HXR)bT9Ee=GhEgLJMF6P|RZ2&8<`M5;nBl'
            '!A6A*TYrx8D$U{&7t0`hH?-jOqG=}E$Dg8Rx??c9j*BebuacO^H5XApHIfa>9GcP#6b)uYV3LC`#CsHkpr}y+Om2!Y0b8XmS+anZ'
            'LU{%z%~gtqPVSg{#?o2KESnBq{q*(&5N`SF+jlP=)u6V;dWYnvU!XbuzLHfLADiD;$L-S0{$$0JNWfVCVins>OlohFUTrG^EZ0C9'
            '{0$5YZcf$?{t~YW5RzmBnC=-UMSQgd@)<P<b_3}_qzVle=U4pODk};`rHro-2#m5VLD3s<={C-JbXCwii>%)B&!i?0D!z^t{&JOm'
            '&PWRa?6suEvC3itsghtZ><E!vOe4wPLB(%LSywSm)+$c<3COmtmS}f@5uYr{UX3=J{Xt$WS7|;+b88H&d*g`VF*D*4w4wf+x;1}$'
            '93B1x996;|l<T?JfEihh!4@sqCukm$T7Cocv1<864gSD|?L2PaJaYSw1nOuoc72pVW`M01YfW12;|yd#B7T@Y`R41#&!4{_%Tp^p'
            'O%$|T&%fFI@EL61yb!-W|3#4TR>6u>a{`894w~m`U8dwqJZ6*M>Ob)gx4jbY>Kr!^ZF#aKpj(}z1)i{&uV9f%G$fe%LNnCV=uV&J'
            'Mi=-`S=jiiVOsi6M;$Py;6|bAw%?(I=0DWpLs^4KAV0xIUKTt0vul_?(3y1yvfO%~c$%YOJmA#{?d-`Ga9pmy3g?|P0Vgy=E0#DL'
            'sX1);Z8+I%viOR&$)hWoA1eBFg4+ccIr5oe<aSg_x3PRR!k(3W>hYM{Zyz6Uh}rylKBKYhgLpps?;dW~8`{TUNy!B*ew5xkelq&<'
            '$z1=nrN6vB>p5A#-~)UDR;2;JSyC)2(Bj;HAI*=`Ps`1oem~aNUq6{00zf1H;&@H1<Le?#mbtuwS%KxublS%|x%@24f+td3H^jtY'
            'h%ydPq)WhmV@Ic=Uj(b$ct}&2Wn|WU49fOEo)&ZlD_gTl1-sATF*FXK^OThqIkbQA3@%ox?&bnx##k}wDit6{74!ucTH7%Mk9XU}'
            '<g3|iHovq64YTy0&y|M%LTVvX<Snl8y0Vr!{{|4{g`@R5r{DFO9@hJ%I{zk<dX!+N-@hhy+W+uH0gWh78m`k_T&DM~2`MQfUMYTr'
            '>WR^1%|js`vgk-vp^c_My5l;4ubZx$$Lm8q?9)sbLaHm+2FF9w0aV6c0SwP_ndX~f4z_!d!MpF{Ov#beO&=_VN;Jr^l;2RCYjVRO'
            'DNqLlPKjt!XW0@d3yUVNC@8_?y`d0~jMrQ90zI|cGb@<R#^OwZvev&M4Ez~iUoC+RftIT?p6FR#EMZi+-Y)s6I77Qkoz2CC2d@D~'
            '4d4g1k5_4yR>!lu%RAqVBR$9yTk46KP=*e@{e;WHdy0c_3-3+TESp)U1^*a(Yq4<uVCooJEzy%03tScePVlA;r$nBRYSsj{Tn0v2'
            '9bd@>nipg6SK~b)vUm!1O`M<;0+5;WEfjnKIt?sbc&8S(Lof%S7Z~1XsT<v}SHrn@I-A{%1(I=z><?7lA5;KtU4Y?DUEp5|eigMM'
            'W>^qwjX5lsrHE?gboOZg6VE`iMjlPL3Bp1qw*HE|n&LCqWnid)6NQEgJS6wbfJ65@Xy*;MEQ=d*^)9XL6M$%B01?1%Z~BmL0G$Xa'
            'Vn~c6UR(ywJ79w^r^KhY1#rQH?mq>NHP7aQ_I?J@lh@LFVKJJ<2aI<{x2=9dyC+~FOLWm9GDo8Z5#XS&Mr}DV)H|rFAH3@8h3}yY'
            'HPd4d**ls#JF;sK=QYx3Hl4wkUk8Bh8gJBH(r$wE5(Ag5j}|?2z%Jcn7$Ubp>yYt+R*rD%0PVc(nDx*XgB|_-23;VCP4!9<&6@Xj'
            'Rx7%yA`I$ylAa-mfZj$j4M_z>R}y@0_=(`g&Rh6>IKnlCA9g##pCr$^AuhX^r^RASkDg&Y5M$!6Za%vkoSd~*>U>7BqyrNbxxS@2'
            'p=lI1Y-Z@qp<0x87gc@8<c0Pg>Br?|6ThuNyZ(H9kC0yyED>MDcnl(%VW!S7XL^7$oX=UDfCU65@foW;qpo8b86C^W@j*4Sj~EF-'
            'c&VwUCTWH|n9-4h+E3siUZDmnVC%6{(%EThPpnM$=NiM8`o&nh;0c%qtanX6aPCnC-&#7(!}jPFXqfOa1b;ULj))92%j+KamLTqk'
            'fg<bVdwIlgcf6%zvPA9xLJbC>f%cd3PDCpm(V7d5@>m%BM#fTS6Mx}QFKP^I()3z$Atjf4CN{KpX&-eonZ`-7G}uJ$B%>~f(+)m%'
            'n-SA2f<8dcW%D{3nf2NTWF{~TRD_lT8rF_Vtjc2$XXY)&yVi7I;2Me(e3YNo@=%Fa^aBi!6QGO$Fd#WLh5+@x_XCL+ksZ${1^WO+'
            'zBV=R6>S(Y7wqgWwLCC}o;KBFyi=Z?UloTXFa(}EEX4T?3?wkX5&#gR1=a+P@a^>Ji3rpd00DPhVThBVJw1Oyq9sY!m6akt(gCMq'
            '%tzpLgC&>A1WyeqFf^jzGLu13$J;Y9`M|!;57Rt}%Q8NK(OvFak7$N$hf`dYjc;rS#^ZE4y}Z-Ge~LV>j)G5SV9%I?+XP;Uyb#q*'
            'AqW~BeWK3+1P-ZVWs;U5;2M{Oig_UcA61eqNe`!ZxEr<XW>qf83uT@HOr0JZ8Pka@JdKVEfY(rY(fm5Pn6V*9L~^^MVGZ_gK^Aia'
            'gaT7LzmgI3ivZ?`hgW4EV4eq30rzg4W`UKOCZCY)2=Eosl%QVB=acixW+za8I+tBIyGmU^?@RBz0{qJ;Ua6?*FcDyS?=OaCEr*vQ'
            '@!g`e42%r4L~LAi1Jl=iWsS22xY6Ktd@`%Vup|e>HE!SuqdykkjPATiAZ<Mq#A2bhnTN}&#O#F@RzyYT&;VyOZ&SgQ6g&e&k$yG^'
            '@9}ryVff1(#l)$XKuA2WxJHFyh{b=vTH&%P+E}^)>=W`HIh0I|v*HGrC%xL*wdJN$fE*!z`43^NkyUlnZGs@gx8ht*&Y!fbJ3s;b'
            'd16?NB+AOC-u(3&1~n3oMbtnZ*@*lB<M5oNuw3C%#$_s1)DD|#<HP)YGUzUfYguN%N#Ic~t+U3~6w1BJd%Cm0ZqU(P+4dUS(_ZTe'
            'KMW}#vsQ#)Cdyp-X8^EL-%MxsA*RiCP}@8;r}u37)zf>rZ(u90;`Oc%z&|}D)zCIL$D}SAtMe7D8<>fdM3ms!eggbrk_ISk22Az+'
            'o>Oab^3TcEODml)@Sdi^=L>qd_|#0p0wXKU*I5lZKy1=dReY+??(sf8^6AAMx)L}_!NIxR#3kvhSmZJ;F|UcTl=NbPu)f)1QVt~#'
            '!4wb@0wDlKf+CEm_!S4JC_DwofbP)%8&EKRjJ{dA3ZN^tt(8pJpr&jh40%w*fer<;gaAoZO$;QllG_-gA|(@PU00OBL+wEv!tXrE'
            'QmnH=QS#TesLbI{oQuLHQd|{9f~h}3XQp|fEW@xELxN2Y=8X{rEuPi_!JJ_(nea|O6eZfHbj}d^x+AT*vk%dQWBdyb%D~}!>fxWh'
            '2h-4-{lf9)K{rjz+t&7RY(^d5;Rs~jDMDO@qMwnw;EXn6Ua81Ka6-{~XqaMvKg{0DwlxV8U4SV(`bST}jMX;*y^9+@0K_lCwfxds'
            'D6J1o&xQpqT()R>A#pDCiZ{X+9STlAxTS?|sP$#kW`l;C%;xggiAZF(owd!I(Kzm?Z3i2x^su?G{X<|*Ga#>h^X<Z?p0U(2(1f@)'
            '|262}EnEs;b|QY%gzuQphz~g06O|cA6QF4~vPaDVU9%vNozo+jW;F)J;_z8((t-WA#*FFFp^iZrXX#Zg6U(P9V1wXjz5;CI70zd9'
            '+j?AaWZ8+h_YY$kFq{nI-KBY*joU^g>@c*{>R5`2x5xetV)52&Lv9>ulnxGhIZg4wRMK1DlXWi2j~+&*5Q`oneIZ`T4bJx(a~iLK'
            '^<uLB2;avnY#23?lxPBw`M}mFX<fE!G&HtIrs6fUtyQJdg2`&AtVyEH2Gr5UwlL-h>);m005!a4<3QPt1|1{>HglegJq61h_TWrV'
            '>hQvexDF2m_|X^*9rFAqn*5U_EjfKV&L%{l$@RH=B<thkIb@;qcZ1zS7qRYMgumBE>j7^Id40V%r#pwkI?uz7-sYolwEMtm>GB`6'
            'bN}R4UNo4;4VD-^SO0}U$S$1rjfe%8P7-(2WZrZ;HOA2-5A(DS)8{Ajx;C7_(ss+dNL9c8!CT1pH@+fyi6Q+YjA_kIctZarH|p<p'
            'pze2_P9m&Lu_Y1J?f2wFzcgQx<FJI;;^A|50Pw4)csOAW3w%HA1tupy?B<2N=pmvMFGr-<3H&IN_k2cRCIp6|Os=1tA=7Tb;czn?'
            'ol7F-D8jvXqT|JM`D8Q|FN*yt&A~R%ayh{zt&W<3C*TS-k0?S1+fJ5d!+8mqwq3JPP|Iv^K6uuS!ab9pXl4m0hZ0-6xpFrW(<8y*'
            'oK4PWGe%P2algt^Ty)LGI&<!-lrl*{Z*bl@xP1p87>E3DSLiWdV9aKBjBztJEdexCr5)q;8H{^7DaC;znBQ*fYZPRP$6Dx(-BjxK'
            'Tb>x^dc+DfaYHW7M*zb=N9_)cV1)?Q%K}#D&D#&+3aO75ll)68cpFGc`*x|{QmUnjM!pwHoV3)lZbFwvMi?ir35>F3kE$;ePhk2q'
            'qD7z6J;EQeb62$3;1))#ym2TF=p^%WB7w2C3jk`I6C2{EC?}ZaUaz@tugJAAm3?O!@N7^YbwrgKyQj;M^SnrX#O>xNbSR53-vFLF'
            'KBoB2nG<dmdhPhp;#)Hz3#iF-=*Ymh;KS}q7pu;h7*x2qJ>~V1D{#K0oNj<(7>_Q87tbgEGoF0@KmR%TzyJH}^3jm<wrq~JpP7ND'
            'xowK!s|zpF;L^`dKaF>*t;zJjGj{T+`51kS$sV1#9@NMkzwLbcROTa!eXMpc8IISZ|JMXy#D6o7?P85W-sZy$jFmUwnjzfOPi=4-'
            'Hl7IQF!+2rYan|&`j6&UIMJgA=2tX<qIQ0TWqS;~<9WRQzH1?&W!K}5oCvTkYdTXVIfAPjj;{?2t%|&s;;~2Eup^l?O+Qud9a-@$'
            '#sr%+C3eFn9TALs#^E(*9Xixj1fD@1FufjcRLH2|ld<^c*=Q69PCQJq(SdNiSRV)TFdM_a?ci=1OCjVphq&Ure}R{cZg(fLmoaze'
            'UC-z{QlJNOZ5S-AVE`}*6D2E+O*eic@g<3OZ=s)%%ECS3WzGyt`X#H_ojm$d$zPpu$Q9!jf4R;{K>BeXt;SuGb<<{@D+Ay|0+Mrk'
            'Bz19f0e`bdzrls`cK#*D*PsQy^5=QySXj#UO#_w$JA5{7ZwRuLJ2u#Cdj9kT8FYhw#(~$fargSeB0hB;J37zI6ZvUuPbqTPlJ%a9'
            'p2(wx6J-f)Doa&YrZq$-3SZDTm&jaTijKvTdr8|!_={mUA}HZS3yoGUdQ43jG1|JkV7%&IwqP{epfS_!M~HEg>EXFQuv0c*X_nZ~'
            'p;C%Ij?#DUgL4M#;g~AAvlk$r=&iZ&ga5Xvdm!Kh-nJN8Rwf=3jDv@jy#U=-9b<%Nrrz{ej+nv6c+C3G4!N|TWw$TFQ|3|Y3^wA!'
            'XV*S^qz|MWs(>V#&KR&0N$}GJnqmWVd67cx9xHp-<=m5R%`oUmkf+00^Ujhgy`3<}0L`{R$04m|3X28!AlpL8d<5#QiDP<=eQSzN'
            'B3CN%sb=Ag8Xv*DV)&z$<NckDoSt&MM=isT0V4SvB0Dz`9lNg~iJx~6go(W1kcqIG*cK48_u#yp9Bk+iaIy2e*;oC&r-$MYbz4tl'
            '-?jrzZD1`AXWSReS@b+_!)yn17VSsvZe8J=Mb|^WtH1OP3DF@Z$%$5g!vyG!p7(WX`R|VG)u*y<3o>`_gVji-Zb&9AxerblG;jd<'
            '5kQz&E3|nJ&^B(pAazi*4`8?>2xR!McdaPwL0vksg4r5b+%_G~LZx?hd2#02EN7Q@U_f+$A`ZS6lxOJl!y>M}ADCwz^+MBD(!JmU'
            'YYN~EPizRc;)r$v8!;SaV|lyezGN(|^jz`EQSzRMQ|nHzn{(|z@#E(P2G-9wl}Q7R#1#-o(WRQ6%eygM84~~WWOndkWfV_r@-sO6'
            '=ngFQH5OpeDIat($_f5aDjk+G;rmu+rYFwqOhlN|Z649Zyt9!_hAbr|LoO`(`f8j0y$L+9i?n=GXW7sCb>tj~GoI$6t@EhW=An0M'
            'tD=gt4&mvbJtfaOnvJ>fSk!r1ErvUw=n#)il)_6+o8j#7Ej73kMV@}n<X-BE6R4*Xx&vhi@0jk+sQRw;;@mN0Tq>J0Wle_1u$!9R'
            '%Uk|=afTY<>~cPPlH5_DmnKz#8w_2TB#gY6j~&9v*wG!^qJGkP=g9BI4GAuK{u!4&gWJR>&=`$dH$#J4$qdU}K@JJfYUtiCo;;bu'
            'M$@-MPh26cGw=fW4sif2xG>yz8`wiF!2i-&yw1D854~~PbL(``s=d<7N-Zp#kaQMWo*@uk;!Bnj%{|p~x~FP;ej%Qt?5)!rD{sNp'
            '#Dj_rme$$Iq&XCz3^r)OA`7;hExN|{K4S+U2>VRvFcfTN78JUqPD>E*Rz58YZn4o_mp~W{)>#Zf&y#Uhe!nRu#X%G+CCh6`m6W(V'
            'kvZd{>SIV&g`(mJ!&gPUhIx~ywO*XUSwu=^8*-o;%yb<l(ixcjZhP?5xCd(EOoA_FXZfX>=<orc5LpdLw<SdwwAT5?Y8yX!`VAMP'
            'A$Gx5(3&Zdq*v@A>+)PvZmv}VsUb{Kj99r2%N5`hBWYkk_w#}8DAoB6*#NI@M(k0}b;~JM(TdJzPrjy@$CIx?Vys|(KDRP5)B1om'
            'MMUtttYiItD?jmXkx4anB>Ylew9)f+g+JB#nv<sWi2y76zM|rRNCg1`GvT6G+j<}8Rzu5mioOaSoFdy>ZU_DX*<)3)q#cdo7;BtO'
            'lX`z}T(=M(Onz1<G-9<*Q*&n)J#`BI1h;LK#H+h_g=!HE$0)$&L(GAbn;1}7pf$uvZd1l08rIcj^4Fn9#$c&6zM~GhuIe<ijwqjm'
            '0s*2y*F>1G`gBqk>oQ$YUs4{ySmS}h2wty_Y{BWLzqdNE73TI@BJ(SIqi$&4QK>g$DB9cY4fX7pamAhggUz9ZzHHdOH}e8`7VANw'
            'cpyfY7#QNWkR()t?>7NzB;CYolLn1B&E^r>VC2Lg`%uM0go~N>&JnNzpE*xAxFb!n5*0!8h-xj))e+E}f~*DsOeus#CS!_J#3BRK'
            'OR7V}S<Y0+Y)u6KIrE5X;bD4lohvDG%xzL>zLqdcV|OQ=dBzH);XY<ng3+dHFbHtNy6G%SZq(X+_N>+ea`LiWtJ0x@jw%HOwZ>TP'
            'XDkur0=u)!PK#A)jp^`QaN*Rgnq@kjz-ai>+n-*&`LKNV>X+BAetq?F`7eLNI%uN65)hxC8EFA_QMymDW+jk#2b;D`!J^c-c^Ba9'
            'W<iDrY*SVft-Ziq_3gLl!|}ifjUD7jHbaY3=)ZsU<Es}~=F80E`I~<ehvIWuVd5zmy;ZyyDn4Sp5e!;`z^A-*06S<*3I%jD2;U;c'
            '#?s(m$!KoqfSRE(%0qs*aF=(eZ;H{8u}c$NnKTbaJ~7Fl4q4Zf7q{LRcTO#`XMtC}8SdpZ9>`%sU6hO?sR2eKuSA(E)0{kKBo;L$'
            '($M^{fIz48)5QdI#xkhGytu)Nlosd(^>+c;mw1W0pLn)bTnO}`7hg-6B$y$yM+Io<5<)YZv9+WQrl*({cTpy-3XYF?ijpz*hUcAU'
            'uG3mExd`Wc;h}Ij=y3jQl}UYi@0i}c1I)V9b?oe$^fP!^N3JAj-<mvZGZcz<&&4LwtgHBAC)|#%eo;W**Jwd_P^rg&_(ts{g*u`D'
            '=Hibo5^e#=7NLAYwB^6ZjH(X!YW%@FJU|s|4GMG!a)~TkHY(FC1L-31m+~&$_QUVv&*GfSy!10TKaUL%u-2*+D~$@+2zb<pP6HMP'
            'Ec+@HI_IHUSJ63Skq_tM+&SMg#sU?I!F=A(&KUZstyPWr4NhU(@a<B^L+8RUv(`x9@y3*IgZ2WL2@*_R0rr5Sy|N_r0CuP(Dt6)Q'
            'ss85BGDsOY-JfBE@#=Z7L0b(*-4L2glg|G3dQ6y3(Ta400NO43VDzC7zHiNdj^SE})6=l6U2MOxeta^U#_!?03VV4nao!F_v?LwH'
            '8#@Ym_yPiN%X$y`T3Z?B_OK6~gbarlJ9R6j9qH#@Q)?zt12zY{8pKu80RKW=ZK_{SK?z2t0QR)5)+pHvtbdL&x&@7W8cj|2(Y>Wn'
            'hW(x|fcpb(@u&x;79%kbjqnPfVpn@h4yBQodr!M-$c|yu(~aBE)Nb!QEYRg@M!oIk2eSas!uOONA%tffloe9$`x=^R{%c3n&bc;P'
            'i>1s>ZtM4S`^WZA*D*fjG@tfin=Ei(2B$=fPAKL*jwcfb)GUgMDi{pV_XI;pae8tUXA<w0pyL;BnyQ@Ep}X)338xCPyzzWIvH<zY'
            'a650|qNT7zE*1Qng8BP7t9K%iS81l>)}}tYcUBiF$Z@C@$5!)=dz8`%Ej0Fb_}Jgi58GGcfH&Q8W@w3TQkf-e;PU+Dt}(`MwNc*i'
            'h{uk+66Ta`(<Z8}rF?cv(C*HJ+^KZe*xD_(T36$y$G$<1!3(WYJi3pt5aPL$?(E^zM#=9E>#VCe@6%&ZhF*Xj)L&W}vu*SABH{HG'
            'EtHto`i3?Fi^jAT)IAe+_7wLwfs`a1jS84!4OV|tm;1uXSG{Aqu?w)e^&l=2TGz!6<gut0ueJ(=^Un8ltT?sF$|!w$5evIf>yQ%d'
            '(YnO)!v^f=`M@!|B)lDy1bsi7wYzc;JL*3?E~YV}e3>Nj8jViua{Oini$5J!TvC90@rNvfp8az|Leie<9=n&wtuP<b7JRBOpk`by'
            '?X~PBMyUQj1*RStwrfO8mv$OGB#zfVjCYXSxjX#vY@D<&5_4a0?IC06_*j$qpp1$-7Q5l-A$rIO47Fuum-#+wRoZM}$}s@~YvM&R'
            '&tyq<6c{CdU>G%8s(QcY8}O$WGnA<z0F64WIBo}TmJg-wlyL7PIo3ImDwE1puszW%AE%C}*54nUf}Z!rqB^2YJxkRQVj8-~+r4xS'
            'oJpd1f57SQ!i5D6C>*T%SL9{P5(@0<;X@)=51p|MQ2A$edds-Sa<0fSGB)Z>e{wVAP1=@DpK9ITy{FO;mm>Br_Kr<*xb|pT-&wHv'
            'p_CSQ|F;!vVg2^CWP053rG*CoG&84fLjk=QTmLMueG>RDNN;@*7^*2$efW(9>M<4X*(wCPBF=8&BMZ5+;Dc0WoIR0py9TR?K6qpE'
            'QXa-!jX>9PuZoE=HXn!Ir)mmy_GvZ$tls)4T_LP-J8CuV2*!<D;k966QH?MghNq26<F@Y01#Tww8;c0fi}g1&M(x7L-A9FH@+c!0'
            'x6a6%vo+%H&^VqGo^&a^GmNI(T-?&&e<_1cCLxAVe~#97r*gDL_XgS@j<>m0j2QzUEzGf<YyEa%+{)2y5N_uL|C#tQ^geXmGLD79'
            'v&&_{JEoByoG3&Yz(p2*^@0=?^qEp2Ty7e+8F<<0Bs+YVZOPOPx1*gd3vbTsy^BwLc^)J0U6<>-1Nq$_uIrBH^UeR}yF|@R4@Zh%'
            '>62%ex&=)e=S9O;^r=XY2At+%jz~CHKz|L<MWIWD;6==i$4;+CyI{@f=03MOHKR;81gigVVX$sN0Q_55l(BWb;Ie@Ut_J**w!W^@'
            '7p@LD5`(|NqcEn};unoX7sFvG&er2zWV+ZHfOkC(dip9A`uGpJTDzLobU1Mb)qW{pSz~6JjcrFACF;W>2ELr4DzNcN6<G~jU2HZ='
            'qGN$M)&8PA>qvudOXA0Y+$Qj|$tqSd@u7FDM8^Qp%{E<cS=Eyg=BkKXRreyGYe45mEh=7i4N+|P`=Rj-=eUoa`F1**wp=XNpnz@|'
            '7xu7c<~i0Rlo)fwG7h_GkmTQeWa4_iUdZ3QGKuBAS{9X4wvEN&#jw?gGEnJMp!I6xf7y*efZhA@)%VXo{`g_}>#NuQ`ojn3b83Iq'
            '1=D}P<x=}m&pHpjE`0xY$x8m@4Br~)G=!<d%!#6+eP%NBd-65izSJK)S@wT<rWd&B+Vy1``|`xMXa=Xtv-VcbZxto#uRb(*u=8T7'
            '61lnm`kN}=ZIFJDrl@<P0nPo&;6pnt-{J?*XA?@nA7lRARVF7oo74GFG?$3C38#nj0L~^jEK$xSX}JCmf1bwusPt&wd;$(d0<rH^'
            'UFt0EvGYM9`~>vaskX0pC6xV&jV>Y11-f;a8Rdq2#`eCBa)S|^FFtk)j6cMa8Z;$VQ>Akyb2eY@{|Yx{aVx>)&zjP&;@!tL`m>hL'
            '-#q{EZ|`5fe<tq#@;3wH?_A%=a>8|PaLFuAK%5n(Q%0@AUSt3J5F=eTn?4BctX<DX!Ly<`V2XP1iER3AIVT~T%1(Zc<`?2O!@2bJ'
            'T&yB%$~2kWch2sfawhxevCh38=@0Chu{5{?X4J#OEhR={{X48`tJ706`gSuLj?WCEf!dbT4LJuHZN0-{ON9oAtv<e{Z}5T(CRmpe'
            'E{HEkKl-+}^T}_j;s>^3nq^q=fj*o${WI=9>8}V|ack16oJXUu%|kCgIfmuJ8{@IHesFs<mx!D){U<xz#Mi@tdxeUZ5d&eAA+C>3'
            '^|x^G_JsLRnG-I&_nVo0pUZSVmC3(%nmgW?20M@?0Ep&$<dGeCM`9a)rYzZx)>^AT`_0h*6xrmqt;y~<Da)Cy|A8_uBBnpR)%rYJ'
            'P@5<GPTQhAY@;Idx4?Ww8GIA0OHq!4XgML*`fk=2^igj5h_+J@E#*31o4ATIa0jrWWxAfw&Up;GV4};gu{z<cw2jRc2Xh^-K=4OD'
            '*`DH3QMOhad#_rSJ5}M%lr2@MVVS~nB)^~wSfn5hIH#Pk)-^V$t`ipDeElBg3%&5>M3LLdbYC>ywOb-FzK+uj(Oqb@s<0?H0L=Nm'
            'iQsllK+VW8n6G51BPACoM_~-o0<Z$4h$7c0|3G`hS#ecMw5L(vP#W}99|W~AG2w*vC&zR&rmK^i=7|SLXCMFe{M~EqQ}1X^F}FM^'
            '@}w5guE=XJ%z$X>kXC}uGr-?SL1zEp6#d&3C1%9JZaNDLmNOM665}ygt`{itJ8rp0`Xe?wsrC|WkRyE`Jw3vlj`0ByUB|m;ob{MN'
            '@0})Y8#+_*f1Tk~Z6Mzf>4!xGbIv6{BB1G6=jLYZNV32nlU9fYN5t!98*@waAMX9`(!w_;-LKjBWtp-5D(dB>|5+DxfnU|Q<a1dX'
            'l@!RuW&7?b6%%3&KNj8}e65Q}+-3GUH`<(TcX~}|@<~1Hp&4y5BS7tfI%%B53Z*$qU@4}B-RaNNjGC{TjP(~%n%)=s>lc?}?;(By'
            '<8shh48p6D*VYYV?;|BZb6b{*o-`GyMy3d`YojzK6R?fcUm5_(3QL}@!52`E&u7YtBvZ%?L1<uf(+8ISg|=FZ)geC}3fwA<1IxGg'
            'hG_1@_$2@;4?=R?8a60SO!-5qC``VAamqp6*08-mvRb)8`dqxVMgC*)WM*V?bEA?zvy86K;wgzQuzE2GfJ%7+xHVl?Vj~GH9hkK#'
            'ZESO`FeN8hdSAN7N;T(WI=d#|T4GsO*8k)S*CRDDt>5S2sx2)Z_^zuKqphIG6=U1I9(UsI#kxBu_nn))<hy;CaCW-dBz@AP6YNBs'
            'pSp1C4T98lD_EYW0{Mmb9(MpT=%GO0HYw8$@J_{k2UZ)fB{Q4y@M>GsI-88*CH{?DjlI|aq^(y82$*pL#w6_l;5hv;7d8kuOzZ#('
            '3Y=QzM%DmZsm3AVby+CmWAfcr=IfsI=K4=h&AxXC@77HZNa^-)%6~RJdf8U@_R5BeFF~OEb<|Zqsa>5p{MMv8U5|O(RSu^w(#pG<'
            '{_jRJ7B1&BXfL_~<R?oSExO6#T9evB={zp%_yY0}oHe{D=G3kQVQ_R$uLUb);06%Nsfs}<9b>dRz(~dmJ9I5$)UDTa;6Rb3e;d|~'
            'XzqIJ&tkJ-;u!k(mJR8x7;;ZxkftyMD2piY&YU_Cb^@b^RT|;xw@;H!obz|xC+<GvUp&ZLa!ivy#DeBPW)Z|^9WTc7AQ$)N1uUG`'
            'gYHy-#l7z4;>0mUE9SfikB*G!qzwPI5@_8`z{7ZI9Wra!(iG~ChJgE(tR`enOpGeU^8BY&Yac(6fi&EtpJX!OrW3pE5$dS#1SvS('
            '@ybfI)Ky1KR_}Pv`;@O+|G3r7JM;4^##+JN(ak8A4pL(Yjnj*D=Zd`MbfU{IqXUYr7yW(hM{R?yXd82ql5H`%ABy*l+It~-t^a`2'
            '*`VMbFJFpB3o(2qetrJ#&FeS+>Ynk68@ysnE_{S1N~W-TMOj&+VbAf6cVVI<AvP$@(0|dvshqCw<iln`GnxUJTMQASnCek=k@EK#'
            'H>Gb-#`)0=7jRf0(o<}r1|tp;;AbRNXee;h8ckg?^Gmq_hd@xlr(!KRj1dD3GQ9UgW?0sL*}|!^8JW}p@1}!$Am``bVK<$`NmM*}'
            'l>jkSpq-8wItDj_E)LAa8P+VlJM%MX(Z{wIb-eL=?uS@CgiA#Ek+@0S4La*hh&yuzIHRBkSqzk^>E*ZhHn&NYFshf*dWf!^dpQ;l'
            'PjI3;bz^&t&O7-V`rmy|lTY2Xrp#pLtvZseYau?$?F6UrvIsYycWQuenH^$Y+yb(9m+d<vxc;ekbHvXz{C~*6M5+'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
